# EA3 — Proyecto final: procesamiento distribuido y cierre del caso

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *41* |
| **Integrantes** | Johnathan Andres Velez · Yuleny Pemberty Espinosa |
| **Caso de estudio** | *Wanderbricks* |
| **Fecha de entrega** | domingo 27 de septiembre |
| **🎥 Enlace al video** | *https://drive.google.com/file/d/1ovoeGFPmtGYRmbvwAfcupB4nGLmnCh0E/view?usp=drive_link* |


---
## 1. Contexto y problema

Esta es la tercera y última entrega del caso Wanderbricks. En la EA1 diseñamos el modelo de datos (arquitectura Lakehouse) y limpiamos `reviews` y `clickstream`. En la EA2 desplegamos el gobierno del entorno — capas bronce/plata/oro en Unity Catalog, permisos con `GRANT`, linaje y un Job automatizado — pero lo hicimos sobre un dataset distinto al de la EA1: usamos `samples.tpch` (proveedores, países, líneas de pedido) en vez de Wanderbricks, porque necesitábamos una tabla lo bastante grande para que la automatización y el gobierno tuvieran sentido de demostrar.

Esa decisión nos dejó dos hilos de datos separados dentro del mismo catálogo `bigdata_grupo41`: uno de negocio turístico (Wanderbricks: países, destinos, anfitriones) y otro de abastecimiento (TPC-H: proveedores, entregas). En esta entrega no unificamos ambos hilos en una sola tabla — no tiene sentido de negocio cruzar anfitriones de Wanderbricks con proveedores de TPC-H — pero sí les damos un cierre a cada uno donde corresponde:

* **Cierre de negocio (Wanderbricks):** construimos la capa oro que nunca llegamos a documentar formalmente en la EA1 — una tabla que responde directamente una de las preguntas de negocio que planteamos desde el principio: ¿en qué países debería el equipo de marketing invertir para conseguir más anfitriones?
* **Cierre técnico (TPC-H):** usamos el cruce más grande que tenemos disponible (`lineitem` completo, sin el muestreo del 2% que usamos para el Job) para medir de verdad una operación costosa, aplicar una optimización y explicar por qué cambió el tiempo de ejecución. Elegimos este hilo para la Parte 2 porque es, con diferencia, el que tiene volumen suficiente para que una optimización se pueda medir sin que el ruido de arranque de Spark tape la diferencia.

El resto del notebook sigue el orden que pide la guía: cerrar el pipeline (capa oro + recorrido del dato), medir y optimizar una operación distribuida, y cerrar el caso respondiendo la pregunta orientadora del curso con conclusiones que se apoyen en los tres proyectos, no solo en este último.

---
## 2. Descripción de los datos

**Hilo Wanderbricks (fuente de la capa oro de esta entrega):**

| Tabla origen | Filas (EA1) | Contenido |
|---|---|---|
| `samples.wanderbricks.countries` | 168 | catálogo de países: nombre, código, continente |
| `samples.wanderbricks.destinations` | 42 | destinos turísticos, cada uno asociado a un país |
| `samples.wanderbricks.hosts` | 19.384 | anfitriones: país, si está verificado, si está activo, calificación |

Estas tres tablas se exploraron en la EA1 pero nunca se llevaron a bronce/plata/oro — solo `reviews` y `clickstream` lo hicieron. En la sección 4 de esta entrega construimos por primera vez esas capas para `countries`, `destinations` y `hosts`, reutilizando el mismo patrón de limpieza de la EA1 (deduplicar, quitar nulos en llaves, `trim` de texto).

**Hilo TPC-H (fuente de la Parte 2 — medición y optimización):**

| Tabla | Filas | Rol |
|---|---|---|
| `bigdata_grupo41.bronce.lineitem` | 599.820 (2% muestra) | la que alimenta el Job automatizado de la EA2 — no se toca aquí |
| `samples.tpch.lineitem` | ~6.000.000 (tabla completa) | la usamos *solo para medir*, sin escribirla a ninguna tabla, porque el 2% de muestra es demasiado pequeño para que un cruce tarde algo medible |
| `samples.tpch.supplier` | 10.000 | tabla pequeña, candidata natural a `broadcast` |
| `samples.tpch.nation` | 25 | tabla pequeña, candidata natural a `broadcast` |

**Frecuencia y consumidores:** igual que en la EA2 — carga batch nocturna para el pipeline gobernado; el cruce de medición de la Parte 2 es puntual (solo para este análisis, no forma parte del Job). Los consumidores de la nueva capa oro son el equipo de marketing y expansión (los mismos de la pregunta de negocio original de la EA1).

---
## 3. Decisiones de diseño y justificación

### 3.1 Dónde vive la nueva capa oro

En vez de retomar el esquema plano `bigdata_grupo41.wanderbricks` que usamos en la EA1 (donde `reviews_bronce`, `reviews_plata` y `clickstream_plata` conviven en un solo esquema), construimos `hosts`, `countries` y `destinations` dentro de los esquemas **`bronce`**, **`plata`** y **`oro`** que ya existen en el catálogo desde la EA2. Razón: esos esquemas ya tienen gobierno aplicado (`GRANT SELECT` sobre `oro` para todos los usuarios, `GRANT MODIFY` sobre `plata` restringido al equipo de datos). Si creamos la tabla nueva ahí, hereda esos permisos automáticamente y no hay que repetir ningún `GRANT`. Mantener dos convenciones de esquema (`wanderbricks` para EA1 y `bronce/plata/oro` para EA2) habría sido inconsistente para alguien que audite el catálogo completo.

La tabla `oro.mercados_principales` ya existía como tabla residual de la EA1 (así la documentamos en los resultados de la EA2), pero nunca se construyó explícitamente con trazabilidad hacia una capa bronce/plata propia. En esta entrega la reconstruimos desde cero a partir de las capas que sí documentamos aquí, con `mode("overwrite")`, para que quede completamente trazada dentro de este pipeline.

### 3.2 Diseño de la capa oro

**Pregunta de negocio (la misma planteada en la EA1, sección 1):** ¿en qué países tenemos mayor cantidad de destinos turísticos, en cuáles se concentran los anfitriones, y en consecuencia dónde debería el equipo de marketing invertir para conseguir más alojamientos?

`oro.mercados_principales` responde esto con una fila por país (no por identificador: el nombre del país es la llave visible), con las métricas ya calculadas: número de destinos, número de anfitriones, porcentaje de anfitriones verificados, calificación promedio, y una bandera `oportunidad_marketing` que marca los países con destinos turísticos pero pocos anfitriones (menos de 40 anfitriones por destino) — que es exactamente la pregunta que el negocio quiere responder rápido y no puede hoy. La tabla queda ordenada con esos países arriba.

### 3.3 Operación elegida para medir y optimizar

Elegimos el cruce `lineitem ⋈ supplier ⋈ nation` sobre la tabla `lineitem` **completa** (no el 2% de muestra), porque:

* Es, con diferencia, la tabla más grande de todo el proyecto (~6 millones de filas frente a las 168 de `countries` o los 19.384 `hosts`); un cruce sobre Wanderbricks habría terminado en milisegundos y la diferencia entre "optimizado" y "sin optimizar" se habría perdido en el ruido de arranque de Spark, como advierte la guía.
* `supplier` (10.000 filas) y `nation` (25 filas) son exactamente el patrón de libro de texto para difusión (*broadcast*): tablas de dimensión pequeñas cruzadas contra una tabla de hechos grande. Es una técnica bien acotada, fácil de explicar y fácil de revertir para comparar.

La técnica elegida es **broadcast join**: en vez de barajar (*shuffle*) las ~6 millones de filas de `lineitem` por la red para que coincidan con las llaves de `supplier`/`nation`, le decimos a Spark que copie las tablas pequeñas completas a cada executor y resuelva el cruce localmente, sin mover la tabla grande.

---
## 4. Implementación
### 4.1 Configuración

In [0]:
from pyspark.sql import functions as F
import time
import statistics

CATALOGO = "bigdata_grupo41"

spark.sql(f"USE CATALOG {CATALOGO}")
print(f"Trabajando en el catálogo {CATALOGO}")

### 4.2 Capa bronce — hosts, countries, destinations

Mismo patrón que usamos en la EA1 para `reviews`: copia cruda de la fuente + metadatos de control (`_ingesta_ts`, `_origen`), sin ninguna limpieza todavía.

In [0]:
fuentes = {
    "hosts_bronce":        "samples.wanderbricks.hosts",
    "countries_bronce":    "samples.wanderbricks.countries",
    "destinations_bronce": "samples.wanderbricks.destinations",
}

for tabla_destino, tabla_origen in fuentes.items():
    (spark.table(tabla_origen)
        .withColumn("_ingesta_ts", F.current_timestamp())
        .withColumn("_origen", F.lit(tabla_origen))
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOGO}.bronce.{tabla_destino}"))
    filas = spark.table(f"{CATALOGO}.bronce.{tabla_destino}").count()
    print(f"{tabla_destino:<22} {filas:>8,} filas")

### 4.3 Capa plata — limpieza de hosts, countries, destinations

Reglas de limpieza (las mismas que en la EA1: deduplicar por llave, quitar nulos en campos obligatorios, `trim` de texto). Cada tabla imprime cuántas filas tenía en bronce, cuántas quedaron en plata y a qué se debe la diferencia — así queda documentado el recorrido del dato incluso si, al ser datos de muestra ya curados por Databricks, la diferencia termina siendo cero.

In [0]:
def limpiar(df, llave, columnas_texto):
    antes = df.count()
    df_limpio = df.dropDuplicates([llave]).dropna(subset=[llave])
    for c in columnas_texto:
        df_limpio = df_limpio.withColumn(c, F.trim(F.col(c)))
    despues = df_limpio.count()
    return df_limpio, antes, despues


# countries_plata
df_countries = spark.table(f"{CATALOGO}.bronce.countries_bronce")
df_countries_plata, antes, despues = limpiar(df_countries, "country", ["country", "country_code", "continent"])
df_countries_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.plata.countries_plata")
print(f"countries_plata     bronce={antes:>8,}  plata={despues:>8,}  descartadas={antes-despues:>4,} (duplicados/país nulo)")

# destinations_plata
df_destinations = spark.table(f"{CATALOGO}.bronce.destinations_bronce")
df_destinations_plata, antes, despues = limpiar(df_destinations, "destination_id", ["destination", "country", "state_or_province"])
df_destinations_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.plata.destinations_plata")
print(f"destinations_plata   bronce={antes:>8,}  plata={despues:>8,}  descartadas={antes-despues:>4,} (duplicados/id nulo)")

# hosts_plata
df_hosts = spark.table(f"{CATALOGO}.bronce.hosts_bronce")
df_hosts_plata, antes, despues = limpiar(df_hosts, "host_id", ["name", "email", "country"])
df_hosts_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.plata.hosts_plata")
print(f"hosts_plata          bronce={antes:>8,}  plata={despues:>8,}  descartadas={antes-despues:>4,} (duplicados/host_id nulo)")

### 4.4 Capa oro — `mercados_principales`

Responde la pregunta de negocio de la sección 3.2: por país, cuántos destinos, cuántos anfitriones, qué tan verificados/bien calificados están, y si es un país con destinos pero pocos anfitriones (oportunidad de marketing). Nombres de país en vez de códigos, métricas ya calculadas, y ordenado con lo más importante arriba.

In [0]:
UMBRAL_ANFITRIONES_POR_DESTINO = 40  # regla de negocio: por debajo de esto, hay hueco de oferta

destinos_por_pais = (spark.table(f"{CATALOGO}.plata.destinations_plata")
    .groupBy("country")
    .agg(F.count("*").alias("num_destinos")))

hosts_por_pais = (spark.table(f"{CATALOGO}.plata.hosts_plata")
    .groupBy("country")
    .agg(
        F.count("*").alias("num_hosts"),
        F.sum(F.col("is_verified").cast("int")).alias("hosts_verificados"),
        F.round(F.avg("rating"), 2).alias("rating_promedio_hosts"),
    ))

paises = spark.table(f"{CATALOGO}.plata.countries_plata")

df_oro = (paises
    .join(destinos_por_pais, paises.country == destinos_por_pais.country, "left")
    .drop(destinos_por_pais.country)
    .join(hosts_por_pais, paises.country == hosts_por_pais.country, "left")
    .drop(hosts_por_pais.country)
    .select(
        F.col("country").alias("pais"),
        F.col("continent").alias("continente"),
        F.coalesce(F.col("num_destinos"), F.lit(0)).alias("num_destinos"),
        F.coalesce(F.col("num_hosts"), F.lit(0)).alias("num_hosts"),
        F.coalesce(F.col("hosts_verificados"), F.lit(0)).alias("hosts_verificados"),
        F.col("rating_promedio_hosts"),
    )
    .withColumn(
        "pct_hosts_verificados",
        F.when(F.col("num_hosts") > 0, F.round(F.col("hosts_verificados") / F.col("num_hosts") * 100, 1)).otherwise(F.lit(None))
    )
    .withColumn(
        "anfitriones_por_destino",
        F.when(F.col("num_destinos") > 0, F.round(F.col("num_hosts") / F.col("num_destinos"), 2)).otherwise(F.lit(None))
    )
    .withColumn(
        "oportunidad_marketing",
        (F.col("num_destinos") > 0) &
        (F.coalesce(F.col("anfitriones_por_destino"), F.lit(0)) < UMBRAL_ANFITRIONES_POR_DESTINO)
    )
    .orderBy(F.col("oportunidad_marketing").desc(), F.col("num_destinos").desc())
)

(df_oro.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.oro.mercados_principales"))

print(f"oro.mercados_principales: {spark.table(f'{CATALOGO}.oro.mercados_principales').count():,} filas (una por país)")
display(spark.table(f"{CATALOGO}.oro.mercados_principales").limit(15))

### 4.5 Recorrido completo del dato (bronce → plata → oro)

In [0]:
print("--- Hilo Wanderbricks (capa oro de esta entrega) ---")
for capa, tabla in [("bronce", "hosts_bronce"), ("plata", "hosts_plata")]:
    print(f"{capa}.{tabla:<20} {spark.table(f'{CATALOGO}.{capa}.{tabla}').count():>8,} filas")
for capa, tabla in [("bronce", "countries_bronce"), ("plata", "countries_plata")]:
    print(f"{capa}.{tabla:<20} {spark.table(f'{CATALOGO}.{capa}.{tabla}').count():>8,} filas")
for capa, tabla in [("bronce", "destinations_bronce"), ("plata", "destinations_plata")]:
    print(f"{capa}.{tabla:<20} {spark.table(f'{CATALOGO}.{capa}.{tabla}').count():>8,} filas")
print(f"oro.mercados_principales    {spark.table(f'{CATALOGO}.oro.mercados_principales').count():>8,} filas (1 por país; agrega, no descarta)")

print()
print("--- Hilo TPC-H (Job automatizado, EA2 — sin cambios en esta entrega) ---")
for capa, tabla in [("bronce", "lineitem"), ("plata", "entregas")]:
    print(f"{capa}.{tabla:<20} {spark.table(f'{CATALOGO}.{capa}.{tabla}').count():>8,} filas")

---
### 4.6 Parte 2 · Selección de la operación costosa

Cruce `lineitem ⋈ supplier ⋈ nation`, replicando la misma lógica de negocio de `plata.entregas` (EA2, sección 4.4) pero sobre `samples.tpch.lineitem` **completo**, sin el muestreo del 2%. No se escribe a ninguna tabla: es un `DataFrame` que se reconstruye cada vez que se mide, precisamente para que la medición sea honesta y repetible.

In [0]:
def construir_entregas_completas():
    """Misma transformación de negocio que bigdata_grupo41.plata.entregas (EA2),
    pero sobre samples.tpch.lineitem sin muestrear, para tener un cruce que sí tarde
    algo medible."""
    lineas = spark.table("samples.tpch.lineitem").alias("l")
    prov   = spark.table("samples.tpch.supplier").alias("s")
    paises = spark.table("samples.tpch.nation").alias("n")

    return (lineas
        .join(prov,   F.col("l.l_suppkey")   == F.col("s.s_suppkey"),   "inner")
        .join(paises, F.col("s.s_nationkey") == F.col("n.n_nationkey"), "inner")
        .filter(F.col("l_extendedprice") > 0)
        .select(
            F.col("l.l_orderkey").alias("pedido_id"),
            F.col("s.s_suppkey").alias("proveedor_id"),
            F.trim(F.col("s.s_name")).alias("proveedor"),
            F.trim(F.col("n.n_name")).alias("pais"),
            F.col("l.l_commitdate").alias("fecha_comprometida"),
            F.col("l.l_receiptdate").alias("fecha_recibido"),
            F.col("l.l_extendedprice").alias("precio"),
            F.col("l.l_discount").alias("descuento"),
        )
        .withColumn("monto_neto", F.round(F.col("precio") * (1 - F.col("descuento")), 2))
        .withColumn("dias_retraso", F.datediff("fecha_recibido", "fecha_comprometida"))
        .withColumn("entrega_tardia", F.col("dias_retraso") > 0)
    )

print(f"lineitem (completo): {spark.table('samples.tpch.lineitem').count():>10,} filas")
print(f"supplier:            {spark.table('samples.tpch.supplier').count():>10,} filas")
print(f"nation:              {spark.table('samples.tpch.nation').count():>10,} filas")

### 4.7 Plan de ejecución inicial (sin optimizar)

Forzamos que Spark **no** use difusión automática (`autoBroadcastJoinThreshold = -1`), para que el plan inicial sea honestamente un *sort-merge join* con *shuffle* de las dos tablas — el punto de partida que vamos a mejorar.

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)  # desactiva la difusión automática: obliga a barajar (shuffle) ambos lados del cruce

df_sin_opt = construir_entregas_completas()
df_sin_opt.explain(mode="formatted")

**Dónde está el costo (para completar después de correrlo):** en el plan de arriba busquen los nodos `Exchange` justo antes de cada `SortMergeJoin` — esos `Exchange` son el *shuffle*: Spark reparte por red las ~6 millones de filas de `lineitem` (y las de `supplier`) agrupándolas por la llave del cruce antes de poder ordenarlas y unirlas. Ese movimiento de datos es el costo que la optimización de la sección 4.9 va a eliminar. *(Peguen aquí, en una línea, el número de `Exchange` que cuentan en su plan real.)*

### 4.8 Medición inicial (calentamiento + 3 mediciones)

Una ejecución de calentamiento (se descarta), después tres mediciones con `time.perf_counter()` sobre un `.count()` que fuerza a Spark a ejecutar todo el plan — no solo a planearlo — y se reporta la mediana.

In [0]:
def medir(construir_df, etiqueta, n=3):
    # Calentamiento: se descarta. Sirve para que Spark resuelva el plan, lea metadatos
    # de las tablas fuente y "caliente" el cluster antes de medir en serio.
    construir_df().count()

    tiempos = []
    for i in range(1, n + 1):
        inicio = time.perf_counter()
        filas = construir_df().count()
        duracion = time.perf_counter() - inicio
        tiempos.append(duracion)
        print(f"  [{etiqueta}] medición {i}: {duracion:6.2f} s   ({filas:,} filas)")

    mediana = statistics.median(tiempos)
    print(f"  [{etiqueta}] mediana de {n} mediciones: {mediana:.2f} s")
    return mediana


print("Mediciones SIN optimizar (sort-merge join forzado):")
mediana_inicial = medir(construir_entregas_completas, "sin optimizar")

### 4.9 Aplicar la optimización: *broadcast join*

En vez de barajar `lineitem` (la tabla grande), le pedimos a Spark que copie `supplier` (10.000 filas) y `nation` (25 filas) completas a cada executor con `F.broadcast(...)`, y resuelva el cruce localmente contra cada partición de `lineitem`, sin moverla. Usamos el *hint* explícito en vez de solo restaurar el umbral automático, para que la optimización quede documentada en el propio plan sin depender de un valor de configuración que alguien podría cambiar después.

In [0]:
def construir_entregas_completas_broadcast():
    """Misma lógica que construir_entregas_completas(), con supplier y nation
    difundidas (broadcast) explícitamente en vez de barajadas."""
    lineas = spark.table("samples.tpch.lineitem").alias("l")
    prov   = F.broadcast(spark.table("samples.tpch.supplier")).alias("s")
    paises = F.broadcast(spark.table("samples.tpch.nation")).alias("n")

    return (lineas
        .join(prov,   F.col("l.l_suppkey")   == F.col("s.s_suppkey"),   "inner")
        .join(paises, F.col("s.s_nationkey") == F.col("n.n_nationkey"), "inner")
        .filter(F.col("l_extendedprice") > 0)
        .select(
            F.col("l.l_orderkey").alias("pedido_id"),
            F.col("s.s_suppkey").alias("proveedor_id"),
            F.trim(F.col("s.s_name")).alias("proveedor"),
            F.trim(F.col("n.n_name")).alias("pais"),
            F.col("l.l_commitdate").alias("fecha_comprometida"),
            F.col("l.l_receiptdate").alias("fecha_recibido"),
            F.col("l.l_extendedprice").alias("precio"),
            F.col("l.l_discount").alias("descuento"),
        )
        .withColumn("monto_neto", F.round(F.col("precio") * (1 - F.col("descuento")), 2))
        .withColumn("dias_retraso", F.datediff("fecha_recibido", "fecha_comprometida"))
        .withColumn("entrega_tardia", F.col("dias_retraso") > 0)
    )

df_opt = construir_entregas_completas_broadcast()
df_opt.explain(mode="formatted")

**Qué cambió en el plan (para completar después de correrlo):** busquen `BroadcastHashJoin` en vez de `SortMergeJoin`, y `BroadcastExchange` (que mueve solo las tablas pequeñas) en vez del `Exchange` que antes movía `lineitem`. *(Peguen aquí si el `Exchange` de la tabla grande desapareció por completo en su plan real.)*

### 4.10 Medición optimizada (mismo método)

In [0]:
print("Mediciones OPTIMIZADAS (broadcast join):")
mediana_optimizada = medir(construir_entregas_completas_broadcast, "optimizado")

---
## 5. Resultados

### 5.1 Comparación de tiempos

In [0]:
mejora_absoluta = mediana_inicial - mediana_optimizada
mejora_pct = (mejora_absoluta / mediana_inicial * 100) if mediana_inicial > 0 else 0.0

print(f"Mediana SIN optimizar:   {mediana_inicial:.2f} s")
print(f"Mediana OPTIMIZADA:      {mediana_optimizada:.2f} s")
print(f"Diferencia:              {mejora_absoluta:+.2f} s")
print(f"Mejora relativa:         {mejora_pct:+.1f} %")

### 5.2 Por qué mejoró (o no) — comparando los dos planes

*Completar con los números y los planes reales de las secciones 4.7 y 4.9. Guía de lo que debería verse y cómo interpretarlo:*

El plan sin optimizar usa `SortMergeJoin`: antes de poder unir `lineitem` con `supplier` y con `nation`, Spark tiene que **barajar (shuffle)** ambas tablas — repartirlas entre executors agrupando las filas que comparten la misma llave (`suppkey`, `nationkey`), para poder ordenarlas y luego recorrerlas en paralelo. Ese `Exchange` es el nodo más caro del plan porque mueve las ~6 millones de filas de `lineitem` por la red (o por disco, si no caben en memoria), aunque las tablas contra las que se cruza sean minúsculas.

El plan optimizado usa `BroadcastHashJoin`: como `supplier` (10.000 filas) y `nation` (25 filas) caben cómodamente en la memoria de cada executor, Spark las copia completas a todos los nodos (`BroadcastExchange`, que mueve KB, no millones de filas) y cada executor resuelve el cruce contra su porción de `lineitem` sin necesidad de barajarla. El `Exchange` de `lineitem` desaparece del plan por completo — esa es la operación que "desapareció" y por qué eso importa: se eliminó el movimiento de la tabla grande, que era el costo real, y solo quedó el movimiento (mucho más barato) de las tablas pequeñas.

*Si al ejecutar esto la mejora medida fue pequeña o nula: eso también es una conclusión válida (ver la advertencia de la sección 5.3) — en un cluster de un solo nodo no hay shuffle "de red" entre executors distintos que eliminar, así que el beneficio del broadcast es mucho menor que en un cluster multi-nodo real. Repórtenlo con los números reales, no lo escondan.*

### 5.3 Cuándo esta misma optimización sería contraproducente

* **Si la tabla "pequeña" no lo es tanto:** `broadcast()` fuerza a Spark a copiar la tabla completa a la memoria de cada executor. Si esa tabla en realidad pesa varios cientos de MB o GB (por ejemplo, si en vez de `nation` con 25 filas difundiéramos `lineitem` completa por error), el resultado no es más velocidad sino errores de memoria (`OutOfMemory`) o *spill* a disco — el broadcast pasa de ser gratis a ser más caro que el shuffle que reemplazaba.
* **En un cluster de un solo nodo (típico de Databricks Free Edition/Serverless de un solo worker):** el shuffle que el broadcast evita es, sobre todo, movimiento de red *entre executors*. Con un único nodo ese movimiento ya es mínimo, así que la ganancia del broadcast puede no notarse — la técnica no está mal aplicada, es que el problema que resuelve (red entre nodos) casi no existe en ese entorno. El mismo cruce en un cluster de varios workers debería mostrar una diferencia mucho más clara.
* **Si la tabla pequeña se difunde muchas veces en consultas repetidas sin cachearla:** cada ejecución vuelve a copiarla a todos los executors; en un `join` que se ejecuta con mucha frecuencia dentro de un pipeline, ese costo de red repetido puede superar al del shuffle ocasional que se quería evitar.
* **Si se difunden varias tablas al mismo tiempo:** el *driver* coordina todas las copias, y una sobrecarga de memoria en el driver puede volverse el nuevo cuello de botella, incluso si cada tabla individualmente es pequeña.

---
## 6. Cierre del caso

### 6.1 Pregunta orientadora del curso

*Una empresa recibe datos de su portal web, de Facebook, de Instagram, de TikTok y de WhatsApp Business. ¿Bajo qué paradigma, con qué herramientas y con qué arquitectura debería analizarlos?*

El paradigma que usamos durante todo el curso —Lakehouse— es el que le conviene a esta empresa, por la misma razón que nos convenció en la EA1: los cinco orígenes no llegan con la misma forma. El portal web probablemente entrega tablas más o menos estructuradas (como nuestro `clickstream`, con eventos y metadatos anidados), mientras que Facebook, Instagram, TikTok y WhatsApp Business entregan JSON semiestructurado con esquemas que cambian según la API de cada plataforma y que van a evolucionar con el tiempo. Un modelo puramente relacional obligaría a definir el esquema por adelantado y a migrar cada vez que una API cambie un campo; un NoSQL documental resolvería la ingesta pero complicaría después los cruces analíticos (por ejemplo, "¿qué campaña generó más reservas, cruzando clics del portal con interacciones de Instagram?"). El Lakehouse permite aterrizar todo crudo en bronce sin pelear con el esquema, y aun así consultar con SQL una vez que la capa plata lo tipa — que es exactamente lo que hicimos con la columna `metadata` anidada de `clickstream` en la EA1.

En herramientas, seguiríamos con la misma pila que ya usamos: Databricks como plataforma gestionada (evita que un equipo pequeño tenga que administrar Spark, un metastore y un orquestador por su cuenta — la comparación de la EA2, sección 3.4, aplica igual aquí), Delta Lake como formato de almacenamiento (da ACID y *time travel*, que importa cuando cinco fuentes escriben en paralelo y hay que poder auditar qué llegó de dónde), y Unity Catalog para gobernar quién puede leer cada fuente — porque no todas las fuentes son igual de sensibles: los datos de WhatsApp Business probablemente incluyen información personal de clientes, mientras que las métricas agregadas de TikTok no.

La arquitectura sería la misma medallón que construimos: cada fuente aterriza en su propio esquema de bronce (`bronce.web`, `bronce.facebook`, `bronce.instagram`, `bronce.tiktok`, `bronce.whatsapp`), con ingestas independientes porque cada API tiene su propio ritmo (el portal web puede ser batch nocturno como el nuestro; las redes sociales normalmente entregan por webhook o API casi en tiempo real). La capa plata es donde se limpia y se unifica el formato (por ejemplo, una tabla `interacciones_plata` común con una columna `plataforma` que diga de dónde vino cada fila), y la capa oro es donde se agregan las métricas de negocio que de verdad se consumen — un dashboard de marketing no debería tener que saber que existen cinco fuentes distintas, solo debería ver una tabla ya unificada y con nombres, igual que nuestra `oro.mercados_principales`.

### 6.2 Conclusiones del proyecto completo (EA1 + EA2 + EA3)

**Qué funcionó:** el patrón bronce/plata/oro se sostuvo bien a lo largo de las tres entregas y sobre datos de naturaleza distinta — texto semiestructurado en la EA1 (`clickstream`), tablas relacionales de abastecimiento en la EA2 (`lineitem`/`supplier`/`nation`), y datos de catálogo turístico en esta entrega (`hosts`/`countries`/`destinations`). El *time travel* de Delta (`DESCRIBE HISTORY`, `VERSION AS OF`) funcionó a la primera desde la EA1, y en esta entrega `.explain()` nos permitió confirmar con evidencia (no solo con el número del cronómetro) por qué una optimización realmente cambia lo que Spark ejecuta.

**Qué no funcionó (o no a la primera):** en la EA2 se nos pasó un catálogo mal escrito en una celda y un `GRANT` que terminó dándole permiso de escritura a todos los usuarios de la cuenta en vez de solo al equipo de datos — los corregimos después de revisar el `SHOW GRANTS` con cuidado, y esa revisión se volvió parte del hábito para esta entrega. En la EA1 tuvimos consultas que devolvían tablas vacías por pedirle demasiado a un dataset de muestra pequeño; en esta entrega evitamos ese mismo error midiendo la operación de la Parte 2 sobre la tabla más grande disponible (`lineitem` completa) en vez de sobre el muestreo de 2% que usamos para el Job, precisamente para que hubiera algo medible.

**Qué haríamos distinto si empezáramos de nuevo:** elegiríamos un único dominio de datos desde la EA1 y lo sostendríamos hasta el final — el haber pivotado a TPC-H en la EA2 para tener volumen nos obligó, en esta entrega, a mantener dos hilos de datos separados dentro del mismo catálogo, lo cual funciona pero le resta continuidad narrativa al caso Wanderbricks. También presupuestaríamos la cuota de Databricks Free Edition desde el inicio del proyecto: esta entrega pide correr la misma consulta ocho veces (calentamiento + 3 mediciones, dos veces), y ese gasto de cómputo hay que planearlo con días de anticipación, no la última noche.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Yuleny Pemberty Espinosa |Contexto y problema, descripción de los datos, diseño de la capa oro (dónde vive y qué responde), construcción de las capas bronce y plata de Wanderbricks (hosts, countries, destinations), construcción de la capa oro mercados_principales y el recorrido del dato (bronce→plata→oro) | Secciones 1, 2, 3.1 y 3.2, 4.2, 4.3, 4.4 (incluida la explicación del umbral de oportunidad_marketing) y 4.5, además de la pregunta orientadora completa (6.1) |
| Johnathan Andres Velez |Selección de la operación costosa, medición inicial y optimizada (calentamiento + 3 mediciones + mediana), aplicación del broadcast join, comparación de planes de ejecución y conclusiones del proyecto completo | Sección 3.3, 4.6 a 4.10, 5.1 a 5.3, y 6.2 — además responde las tres preguntas obligatorias de su intervención (plan antes/después, cuándo sería contraproducente, y la pregunta orientadora) |

**Uso de asistentes de IA:** en esta entrega usamos Claude (Anthropic) como apoyo para: redactar y revisar el notebook completo de la EA3 a partir de los notebooks ya entregados de la EA1 y la EA2, El código se revisó y se ejecutó en Databricks antes de entregar.

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] La capa oro (`oro.mercados_principales`) existe y sus columnas se entienden sin explicación
- [ ] Las tres mediciones están visibles, antes y después, con la mediana reportada
- [ ] Los dos planes de ejecución (`.explain()`) están en el notebook, con la salida visible
- [ ] Está escrito por qué mejoró (o no) y cuándo la técnica sería contraproducente
- [ ] La pregunta orientadora está respondida, apoyada en lo construido durante el curso
- [ ] La sección 7 declara qué hizo cada integrante y coincide con lo que cada uno sustenta en el video
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] El notebook `.ipynb` está confirmado en el repositorio, en la carpeta `/ea3`
- [ ] El notebook se ejecutó completo y el HTML con salidas visibles se subió a Canvas (`File → Export → HTML`)